# 3주차 실습. 불확실성 → 결정

2주차까지는 예측값 하나(ŷ)만 봤다. 이번 주는 예측마다 **σ (얼마나 확신하는가)** 를 같이 내고,
그 σ 가 맞는지 재고, σ 를 써서 **다음에 잴 분자**를 고른다.

| 절 | 내용 | 쓰는 함수 (`uncertainty.py`) |
|---|---|---|
| 1 | 데이터와 분할 (2주차 `train.run` 과 같은 분할) | `split_smiles`, `morgan` |
| 2 | σ 를 내는 세 방법: Random Forest 앙상블 · BGNN-GIN · 가우시안 과정(GP) | `rf_sigma`, `train.run(bayes=True)`, `gp_sigma` |
| 3 | σ 가 맞는가: 2σ 적중률, σ 구간별 실제 오차, Spearman | `calibration` |
| 4 | 적용범위: 학습 세트와 닮지 않은 분자에서 오차가 커지나 (random vs scaffold) | `max_similarity` |
| 5 | 결정: 능동학습 시뮬레이션 (무작위 vs 최대 σ vs EI vs UCB) | `active_learning` |
| 6 | 팀 과제 | |

전체 실행 약 4분 (CPU, logP 3,000개). 가장 오래 걸리는 셀은 BGNN 학습(2절).

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')            # GP 최적화 경계 경고 숨김
from uncertainty import (split_smiles, morgan, rf_sigma, gp_sigma, gp_noise_sd, calibration,
                         max_similarity, error_by_similarity, active_learning, check_fig, panel, OKABE)

PROP  = 'logP'        # 팀 물성으로 바꾼다 (data/ 의 파일 이름)
MAX_N = 3000          # 수업 시간용. 집에서는 5000 또는 0(전체)
SEED  = 42
OUT   = 'results_uq'; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({'font.size': 12, 'axes.linewidth': 1.2, 'legend.frameon': False})
T = {}                # 셀별 실행 시간 (초)

## 1. 데이터와 분할
`split_smiles` 는 `train.run` 과 같은 함수(`load_data → gen_dataset → split_dataset`)를 같은 seed 로 부른다.
그래서 RF · GP 의 test 분자와 BGNN 의 test 분자가 **같은 300개, 같은 순서**다. 세 방법의 σ 를 같은 분자에서 비교할 수 있다.

In [ ]:
t = time.time()
d = split_smiles(PROP, MAX_N, 'random', SEED)
Xtr, Xva, Xte = (morgan(d[k].smiles) for k in ('train', 'validate', 'test'))
ytr, yva, yte = (d[k].y.values for k in ('train', 'validate', 'test'))
T['data'] = time.time() - t
print({k: len(v) for k, v in d.items()}, f'| Morgan {Xtr.shape[1]} bits | {T["data"]:.0f} s')

## 2. σ 를 내는 세 방법

| 방법 | σ 의 출처 | 입력 |
|---|---|---|
| Random Forest 앙상블 | 나무 300그루 예측의 표준편차. 나무마다 다른 부트스트랩 표본을 봤다 | Morgan |
| BGNN-GIN | 마지막 선형층 가중치를 분포로 두고 100번 뽑은 예측의 표준편차 (2주차 14쪽) | 분자 그래프 |
| GP (Tanimoto 커널) | 학습 분자와 얼마나 닮았나. 닮은 분자가 없으면 σ 가 커진다. 잡음 항 포함 | Morgan |

In [ ]:
t = time.time(); mu_rf, sd_rf = rf_sigma(Xtr, ytr, Xte, seed=SEED); T['RF'] = time.time() - t
t = time.time(); mu_gp, sd_gp, gp = gp_sigma(Xtr, ytr, Xte, return_model=True); T['GP'] = time.time() - t
print(f'RF {T["RF"]:.0f} s | GP {T["GP"]:.0f} s | GP 커널: {gp.kernel_}')

In [ ]:
# BGNN-GIN : 2주차 train.run 을 그대로 쓴다. 결과는 results_uq/ 에 쓴다 (results/summary.csv 는 건드리지 않음)
from train import run
t = time.time()
res = run(prop=PROP, bayes=True, max_n=MAX_N, seed=SEED, outdir=OUT, verbose=False)
T['BGNN'] = time.time() - t
bg = pd.read_csv(f'{OUT}/{PROP}_BGNN-GIN_mean_random_test_pred.csv')
assert (bg.smiles.values == d['test'].smiles.values).all()      # 같은 test 분자, 같은 순서
mu_bg, sd_bg = bg.pred.values, bg.sd.values
print(f'BGNN {T["BGNN"]:.0f} s, {res["epochs"]} epochs')

## 3. σ 가 맞는가 (보정, calibration)
세 가지를 본다.
1. **2σ 적중률**: |오차| ≤ 2σ 인 test 분자 비율. 오차가 정규분포면 0.954. 낮으면 σ 를 너무 작게(자신만만하게) 잡은 것
2. **σ 구간별 실제 오차**: σ 크기 순으로 8구간으로 나눠 구간마다 평균 σ 와 실제 RMSE 를 찍는다. 대각선 위에 있으면 σ 가 맞다
3. **Spearman(|오차|, σ)**: σ 가 큰 분자일수록 실제로 더 틀리나 (순위 상관, 1 = 완벽)

In [ ]:
rows, bins = [], {}
for name, mu, sd in [('RF', mu_rf, sd_rf), ('BGNN-GIN', mu_bg, sd_bg), ('GP', mu_gp, sd_gp)]:
    m, b = calibration(yte, mu, sd); m['model'] = name; rows.append(m); bins[name] = b
metrics = pd.DataFrame(rows).set_index('model')[['MAE', 'mean_sd', 'coverage_2sd', 'spearman']]
metrics.round(3).to_csv(f'{OUT}/{PROP}_calibration_random.csv')
pd.concat({k: v for k, v in bins.items()}).to_csv(f'{OUT}/{PROP}_calibration_bins.csv')
pd.DataFrame({'smiles': d['test'].smiles, 'y': yte, 'mu_rf': mu_rf, 'sd_rf': sd_rf, 'mu_bgnn': mu_bg, 'sd_bgnn': sd_bg,
              'mu_gp': mu_gp, 'sd_gp': sd_gp}).to_csv(f'{OUT}/{PROP}_test_sigma_random.csv', index=False)
metrics.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
checks = []
for ax, (name, col), tag in zip(axes, [('RF', OKABE[0]), ('BGNN-GIN', OKABE[3]), ('GP', OKABE[2])], 'abc'):
    b = bins[name]; hi = 1.1 * max(b.sd_mean.max(), b.rmse.max())
    ax.plot([0, hi], [0, hi], color='0.5', lw=1, label='σ = RMSE')
    ax.plot(b.sd_mean, b.rmse, 'o-', color=col, label=name)
    ax.set_xlim(0, hi); ax.set_ylim(0, hi)
    ax.set_xlabel(f'mean predicted σ ({PROP})'); ax.set_ylabel(f'observed RMSE ({PROP})')
    ax.legend(loc='upper left'); panel(ax, tag); checks.append((ax, b.sd_mean, b.rmse))
check_fig(fig, checks); fig.tight_layout(); fig.savefig(f'{OUT}/{PROP}_calibration.png', dpi=150); plt.show()

**우연 불확실성과 지식 불확실성.** GP 는 σ 를 두 조각으로 나눠 준다.
- 잡음 항 (WhiteKernel) = 같은 분자를 다시 재도 남는 흩어짐. 데이터를 더 모아도 줄지 않는다 (**aleatoric, 우연**)
- 나머지 = 학습 분자와 닮지 않아서 생기는 몫. 데이터를 모으면 준다 (**epistemic, 지식**)

In [ ]:
noise = gp_noise_sd(gp)
epi = np.sqrt(np.maximum(sd_gp ** 2 - noise ** 2, 0))
print(f'GP 잡음 σ (우연)          = {noise:.3f}')
print(f'test 평균 전체 σ           = {sd_gp.mean():.3f}')
print(f'test 평균 지식 σ (나머지)  = {epi.mean():.3f}')
print(f'Spearman(σ_RF, σ_GP) = {spearmanr(sd_rf, sd_gp).correlation:.2f},  '
      f'Spearman(σ_BGNN, σ_GP) = {spearmanr(sd_bg, sd_gp).correlation:.2f}   # 세 방법이 같은 분자를 불확실하다고 보나')

## 4. 적용범위 (applicability domain)
모델은 학습 세트와 닮은 분자에서만 믿을 만하다. "닮음" = test 분자마다 **학습 세트에서 가장 닮은 분자와의 Tanimoto 유사도**.
scaffold 분할은 test 에 학습에서 못 본 골격만 넣으므로 이 값이 낮아진다. 같은 GP 를 두 분할에서 학습해 비교한다.

In [ ]:
t = time.time()
ds = split_smiles(PROP, MAX_N, 'scaffold', SEED)
Xtr_s, Xte_s = morgan(ds['train'].smiles), morgan(ds['test'].smiles)
ytr_s, yte_s = ds['train'].y.values, ds['test'].y.values
mu_gs, sd_gs = gp_sigma(Xtr_s, ytr_s, Xte_s)
mu_rs, sd_rs = rf_sigma(Xtr_s, ytr_s, Xte_s, seed=SEED)
sim_r = max_similarity(d['train'].smiles, d['test'].smiles)
sim_s = max_similarity(ds['train'].smiles, ds['test'].smiles)
T['AD'] = time.time() - t
err_r, err_s = np.abs(yte - mu_gp), np.abs(yte_s - mu_gs)
ad = pd.concat({'random': error_by_similarity(sim_r, err_r), 'scaffold': error_by_similarity(sim_s, err_s)})
ad.to_csv(f'{OUT}/{PROP}_ad_bins.csv')
pd.DataFrame({'split': ['random'] * len(yte) + ['scaffold'] * len(yte_s), 'max_sim': np.r_[sim_r, sim_s],
              'abs_err_gp': np.r_[err_r, err_s], 'sd_gp': np.r_[sd_gp, sd_gs]}).to_csv(f'{OUT}/{PROP}_ad_points.csv', index=False)
sc_rows = []
for name, mu, sd in [('RF', mu_rs, sd_rs), ('GP', mu_gs, sd_gs)]:
    m, _ = calibration(yte_s, mu, sd); m['model'] = name; sc_rows.append(m)
sc = pd.DataFrame(sc_rows).set_index('model')[['MAE', 'mean_sd', 'coverage_2sd', 'spearman']]
sc.round(3).to_csv(f'{OUT}/{PROP}_calibration_scaffold.csv')
print(f'test 분자의 최대 유사도 중앙값: random {np.median(sim_r):.2f}, scaffold {np.median(sim_s):.2f}')
print(f'유사도 < 0.4 비율: random {np.mean(sim_r < 0.4):.2f}, scaffold {np.mean(sim_s < 0.4):.2f}')
print(f'Spearman(유사도, GP σ) random {spearmanr(sim_r, sd_gp).correlation:.2f}, scaffold {spearmanr(sim_s, sd_gs).correlation:.2f}')
print('scaffold 분할 보정:'); print(sc.round(3)); ad.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
edges = np.linspace(0, 1, 21)
axes[0].hist(sim_r, edges, histtype='step', lw=2, color=OKABE[0], label='random')
axes[0].hist(sim_s, edges, histtype='step', lw=2, color=OKABE[1], label='scaffold')
axes[0].set_xlabel('max Tanimoto to training set'); axes[0].set_ylabel('test molecules'); axes[0].legend(loc='upper left')
for split, col in [('random', OKABE[0]), ('scaffold', OKABE[1])]:
    a = ad.loc[split]; a = a[a.n >= 5]
    axes[1].plot((a.lo + a.hi) / 2, a.mae, 'o-', color=col, label=split)
axes[1].set_xlabel('max Tanimoto to training set (bin centre)'); axes[1].set_ylabel(f'GP mean |error| ({PROP})')
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1.15 * ad[ad.n >= 5].mae.max()); axes[1].legend(loc='upper right')
for ax, tag in zip(axes, 'ab'): panel(ax, tag)
check_fig(fig); fig.tight_layout(); fig.savefig(f'{OUT}/{PROP}_applicability.png', dpi=150); plt.show()

## 5. 결정: 다음에 잴 분자 고르기 (능동학습 시뮬레이션)
실험실 상황을 흉내 낸다. train + validate 2,700개를 **아직 안 잰 후보(pool)** 로 두고 y 를 숨긴다.
처음 5% 만 무작위로 '재고', 라운드마다 5% 씩 골라 더 잰다 (10 라운드, 55% 까지). 모델은 매 라운드 GP 를 다시 학습한다.

| 전략 | 고르는 기준 | 목표 |
|---|---|---|
| random | 무작위 | 비교 기준 |
| sigma | σ 가 가장 큰 분자 | 모델 전체를 정확하게 (MAE) |
| ei | 기대 개선 EI: 지금 최고값을 얼마나 넘을지의 기댓값 | **물성이 가장 큰 분자 찾기** |
| ucb | μ + 2.576σ (연구실 `cjh_bayes_optimization.py` 의 kappa) | 물성이 가장 큰 분자 찾기 |

기록: test MAE, 지금까지 찾은 최고 y, pool 상위 1% (27개) 중 찾은 비율. 출발 표본(seed 0, 1, 2)을 바꿔 3번 반복하고, 선 = 평균, 띠 = 최소–최대.

In [ ]:
t = time.time()
X_pool, y_pool = np.r_[Xtr, Xva], np.r_[ytr, yva]
al = pd.concat([active_learning(X_pool, y_pool, Xte, yte, s, seed=k)
                for s in ['random', 'sigma', 'ei', 'ucb'] for k in range(3)], ignore_index=True)
T['AL'] = time.time() - t
al.to_csv(f'{OUT}/{PROP}_active_learning.csv', index=False)
print(f'{T["AL"]:.0f} s | pool {len(y_pool)}, pool 최댓값 {y_pool.max():.2f}')
g = al.groupby(['strategy', 'n_labeled'])[['test_MAE', 'best_found', 'top_found']].mean()
g.unstack(0).loc[[135, 540, 945, 1485]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
cols = {'random': OKABE[7], 'sigma': OKABE[0], 'ei': OKABE[3], 'ucb': OKABE[2]}
ylab = {'test_MAE': f'test MAE ({PROP})', 'best_found': f'best {PROP} found', 'top_found': 'top-1% of pool found'}
for ax, key, tag in zip(axes, ['test_MAE', 'best_found', 'top_found'], 'abc'):
    for s, col in cols.items():
        q = al[al.strategy == s].groupby('n_labeled')[key]
        ax.fill_between(q.mean().index, q.min(), q.max(), color=col, alpha=0.15, lw=0)
        ax.plot(q.mean().index, q.mean(), 'o-', ms=4, color=col, label=s)
    if key == 'best_found':
        ax.axhline(y_pool.max(), color='0.5', lw=1, ls='--', label='pool max')
    ax.set_xlabel('labeled molecules'); ax.set_ylabel(ylab[key]); panel(ax, tag)
    lo, hi = al[key].min(), max(al[key].max(), y_pool.max() if key == 'best_found' else al[key].max())
    ax.set_ylim(lo - 0.05 * (hi - lo), hi + 0.08 * (hi - lo))
    ax.legend(loc='upper right' if key == 'test_MAE' else 'lower right', fontsize=10)
check_fig(fig); fig.tight_layout(); fig.savefig(f'{OUT}/{PROP}_active_learning.png', dpi=150); plt.show()

**읽을 것**
- a) MAE 가 빨리 내려가는 전략과 b) 최고값을 빨리 찾는 전략이 같은가? 목표가 다르면 고르는 기준도 달라야 한다
- EI · UCB 로 고른 분자로 학습한 모델의 test MAE 는 어떤가? 한쪽(큰 값)만 잰 데이터로 만든 모델을 다른 용도로 쓰면 생기는 문제
- 한 라운드에 135개를 점수 순으로 한꺼번에 고른다. 서로 닮은 분자가 같이 뽑힐 수 있다 (실제 BO 는 배치 다양성을 따로 챙긴다)

In [ ]:
json.dump({'prop': PROP, 'max_n': MAX_N, 'seed': SEED, 'sec': {k: round(v, 1) for k, v in T.items()},
           'total_sec': round(sum(T.values()), 1), 'bgnn_epochs': res['epochs'], 'gp_kernel': str(gp.kernel_),
           'gp_noise_sd': noise}, open(f'{OUT}/{PROP}_run_info.json', 'w'), indent=1)
print({k: round(v) for k, v in T.items()}, '| total', round(sum(T.values())), 's')

## 6. 팀 과제
1. `PROP` 를 팀 물성으로 바꿔 끝까지 돌리고 **3절 표** (MAE, 평균 σ, 2σ 적중률, Spearman) 를 random · scaffold 두 분할로 낸다
2. 2σ 적중률이 0.954 에서 가장 먼 방법은? 그 방법의 σ 를 몇 배 하면 0.954 가 되는지 **validate 세트로** 구하고, 그 배율을 test 에 적용해 다시 잰다 (재보정)
3. 4절: scaffold test 에서 유사도 0.4 미만 분자 5개를 그리고 학습 세트의 가장 닮은 분자와 나란히 놓는다. 왜 틀렸는지 화학적으로 한 줄씩
4. 5절: 팀 물성에서 '가장 큰 값' 대신 **목표 구간**(예: 점도 1–10 cP, pKa 9–10)에 드는 분자를 찾는 획득 함수를 만들어 본다. 힌트: $P(a < y < b) = \Phi(\frac{b-\mu}{\sigma}) - \Phi(\frac{a-\mu}{\sigma})$
5. 토론: 연구실 사례(상속 pKa 예측을 측정과 대조하지 않고 씀)에서 이번 주 도구 중 무엇이 있었으면 문제를 일찍 알았을까?

규칙: σ 재보정 배율, 획득 함수 설정은 validate 로 고른다. test 는 마지막에 한 번만 본다.